In [1]:
import os
import cv2
import torch
import numpy as np
!pip install ultralytics
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import sys
import subprocess


llista_lleida = [
    "01_James_Batemon", "02_Corey_Walden", "03_Melvin_Ejim",
    "04_Caleb_Agada", "05_Oriol_Pauli", "06_Dani_Garcia",
    "07_Gyorgy_Goloman", "12_Atoumane_Diagne", "14_Mikel_Sanz",
    "16_Millan_Jimenez", "22_John_Shurna", "25_Cameron_Krutwig"
]

llista_joventut = [
    "00_H_Drell",
    "01_Y_Kraag",
    "09_R_Rubio",
    "11_J_Parker",
    "12_L_Hakanson",
    "21_M_Allen",
    "23_M_Ruzic",
    "24_C_Hunt",
    "32_F_Mauri",
    "34_D_Niebla",
    "35_S_Birgander",
    "88_A_Hanga"
]

tots_els_jugadors = llista_lleida + llista_joventut

if os.path.exists('/kaggle/working'):
    base_path = '/kaggle/working/dataset_reid'
elif 'COLAB_GPU' in os.environ or 'GCS_READ_CACHE' in os.environ:
    base_path = '/content/dataset_reid'
else:
    base_path = './dataset_reid'

for j in tots_els_jugadors:
    os.makedirs(os.path.join(base_path, 'train', j), exist_ok=True)
    os.makedirs(os.path.join(base_path, 'test/gallery', j), exist_ok=True)
    os.makedirs(os.path.join(base_path, 'test/query', j), exist_ok=True)

print(" Estructura de carpetes Re-ID creada amb èxit.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Estructura de carpetes Re-ID creada amb èxit.


In [1]:
try:
    import yt_dlp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yt-dlp"])

url = "https://www.3cat.cat/3cat/lleida-joventut-lliga-j23/video/6394037/"
output_name = "partit_lleida_V2.mp4"
format_str = "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4"

os.system(f'yt-dlp -f "{format_str}" "{url}" -o "{output_name}"')

ruta_video = "partit_lleida_V2.mp4"

if os.path.exists(ruta_video):
    mida_mb = os.path.getsize(ruta_video) / (1024 * 1024)
    print(f"\n Vídeo descarregat correctament!")
    print(f"   Mida del fitxer: {mida_mb:.2f} MB")
else:
    print("\n ERROR: El vídeo no s'ha descarregat.")

NameError: name 'subprocess' is not defined

In [2]:
detector = YOLO('yolov8x.pt')

estat_crops = []
crop_actual = None
index_actual = 0

def comptar_imatges_actuals():
    estat = {}
    for j in tots_els_jugadors:
        n_train = len(os.listdir(os.path.join(base_path, 'train', j)))
        n_query = len(os.listdir(os.path.join(base_path, 'test/query', j)))
        n_gallery = len(os.listdir(os.path.join(base_path, 'test/gallery', j)))
        estat[j] = {'train': n_train, 'query': n_query, 'gallery': n_gallery}
    return estat

def demanar_moments_video(ruta_video, segons_llista):
    global estat_crops, index_actual
    estat_crops = []
    index_actual = 0

    cap = cv2.VideoCapture(ruta_video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25

    print(f"🕵️‍♂️ Processant {len(segons_llista)} frames del partit...")
    for segon in segons_llista:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(segon * fps))
        ret, frame = cap.read()
        if not ret: continue

        results = detector(frame, classes=0, conf=0.45, verbose=False)[0]
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            y1, y2 = max(0, y1), min(frame.shape[0], y2)
            x1, x2 = max(0, x1), min(frame.shape[1], x2)

            crop = frame[y1:y2, x1:x2]
            if crop.size > 0:
                estat_crops.append(crop)

    cap.release()
    print(f"📸 S'han trobat {len(estat_crops)} retalls. Desplegant panell de control...")
    mostrar_seguent_crop()

def desar_imatge(jugador, crop):
    estat = comptar_imatges_actuals()[jugador]

    if estat['train'] < 20:
        carpeta = f"train/{jugador}"
        num_foto = estat['train']
    elif estat['query'] < 15:
        carpeta = f"test/query/{jugador}"
        num_foto = estat['query']
    elif estat['gallery'] < 1:
        carpeta = f"test/gallery/{jugador}"
        num_foto = estat['gallery']
    else:
        print(f"⚠️ {jugador} ja té totes les seues carpetes plenes!")
        return

    nom_fitxer = os.path.join(base_path, carpeta, f"{jugador}_{num_foto}.jpg")
    cv2.imwrite(nom_fitxer, crop)
    print(f"💾 Desat: {carpeta} (Nº {num_foto})")

output_visual = widgets.Output()

def mostrar_seguent_crop():
    global index_actual, crop_actual
    with output_visual:
        clear_output(wait=True)
        if index_actual >= len(estat_crops):
            print("🏁 S'han revisat tots els retalls d'aquest bloc de segons!")
            return

        crop_actual = estat_crops[index_actual]
        img_rgb = cv2.cvtColor(crop_actual, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)

        print(f"Retall {index_actual + 1} de {len(estat_crops)}")
        display(img_pil.resize((150, 300)))

        estat = comptar_imatges_actuals()
        print("\n📈 Resum d'ompliment (Train [20] / Query [15] / Gallery [1]):")
        for j in tots_els_jugadors:
            v = estat[j]
            if v['train'] > 0 or v['query'] > 0 or v['gallery'] > 0:
                print(f"   • {j}: {v['train']}/20 | {v['query']}/15 | {v['gallery']}/1")

def al_clicar_jugador(b):
    global index_actual
    desar_imatge(b.description, crop_actual)
    index_actual += 1
    mostrar_seguent_crop()

def al_clicar_passar(b):
    global index_actual
    print("⏭️ Element descartat")
    index_actual += 1
    mostrar_seguent_crop()

botons_lleida = [widgets.Button(description=j, button_style='info', layout=widgets.Layout(width='200px')) for j in llista_lleida]
botons_joventut = [widgets.Button(description=j, button_style='success', layout=widgets.Layout(width='200px')) for j in llista_joventut]

boto_passar = widgets.Button(description=" IGNORAR ELEMENT", button_style='danger', layout=widgets.Layout(width='420px', height='45px'))

for b in botons_lleida + botons_joventut:
    b.on_click(al_clicar_jugador)
boto_passar.on_click(al_clicar_passar)

graella_lleida = widgets.GridBox(botons_lleida, layout=widgets.Layout(grid_template_columns="repeat(2, 210px)"))
graella_joventut = widgets.GridBox(botons_joventut, layout=widgets.Layout(grid_template_columns="repeat(2, 210px)"))

pestanyes_equips = widgets.Tab()
pestanyes_equips.children = [graella_lleida, graella_joventut]
pestanyes_equips.set_title(0, 'Hiopos Lleida')
pestanyes_equips.set_title(1, 'Joventut Badalona')

panell_control = widgets.VBox([boto_passar, widgets.Label("👇 Selecciona a quin jugador pertany la silueta:"), pestanyes_equips])

NameError: name 'YOLO' is not defined

In [ ]:
segons_a_analitzar = [100,101,102,103,104,105,107,108,109,110,111,112,113,90,95,3300,54,55,56,3301,3302,3303,4501,4502,199,200, 201, 202, 203, 204, 205]

display(widgets.HBox([output_visual, panell_control]))

demanar_moments_video('partit_lleida_V2.mp4', segons_a_analitzar)